# supervise-all curriculum — 500 problems, 3 schedules, 2 epochs
### coarse → fine, with **every** block position supervised at every sub-step

**The experiment.** The original stage A used `supervise_all=True` at 4 tokens/step, then stages B
and C switched to the standard masked-only objective at 2 and 1 tokens/step. This notebook instead
keeps `supervise_all=True` **throughout** and varies only the reveal schedule:

| stage | tokens/step | objective | data |
|---|---|---|---|
| **S1** | 4 | all 32 block positions supervised each sub-step | 500 problems × 2 epochs |
| **S2** | 2 | same objective, finer reveal | same 500 × 2 |
| **S3** | 1 | same objective, finest reveal | same 500 × 2 |

Each stage resumes from the previous one's weights, so S3's output carries S1+S2+S3.

**What `supervise_all=True` actually trains.** At each sub-step the model predicts every position in
the block, including ones already revealed — and a revealed position has its gold token sitting in
the model's own input at that index, so those terms are close to a copy-through. That makes the
*loss* incomparable to a masked-only stage (it's diluted by easy terms), which is why this notebook
judges progress by **benchmark accuracy after each stage**, never by training loss.

**Why re-run it.** The benchmark showed the original SFT hurt medium/hard problems at both
schedules, with repeat-4 rising monotonically A→B→C. This isolates one variable — keeping the
objective fixed and sweeping only the schedule — over 5× more data than the original stage A.

> **Isolation.** Checkpoints and results write to a **separate** `drive_root`, and training data is
> `stage_a + stage_b + stage_c` (500 problems). The OPSD pool and the 40-problem benchmark are
> untouched, so this can run alongside the OPSD notebook without collision.


## 1 · GPU & Drive

In [ ]:
!nvidia-smi
import torch, platform
print("\nTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| Python:", platform.python_version())
assert torch.cuda.is_available(), "No GPU — Runtime ▸ Change runtime type."
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2 · Dependencies — torchao removed **before** transformers is imported
**If the transformers version changes, restart the runtime and re-run from the top.**

In [ ]:
import os, re, subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
for _n in [n for n in list(sys.modules) if n == "torchao" or n.startswith("torchao.")]:
    del sys.modules[_n]

if not os.path.isdir('/content/dLLM-RL'):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/Gen-Verse/dLLM-RL","/content/dLLM-RL"], check=True)

_FALLBACK = "transformers==4.51.3"; _spec = _FALLBACK
_req = "/content/dLLM-RL/requirements.txt"
if os.path.isfile(_req):
    m = re.search(r"^\s*transformers(\[[^\]]*\])?\s*([=<>!~].*?)\s*(?:#.*)?$", open(_req).read(), re.M)
    if m and m.group(2): _spec = "transformers" + (m.group(1) or "") + m.group(2).strip()
print("Pinning:", _spec)

!pip -q install "{_spec}" "accelerate>=0.33" "peft>=0.12" "datasets>=2.20" sentencepiece packaging

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
import importlib.util
print("torchao present?", importlib.util.find_spec("torchao") is not None, "(must be False)")
import transformers; print("transformers:", transformers.__version__)
print("⚠️ If that version just CHANGED: Runtime ▸ Restart session, then run from the top.")


## 3 · Config

In [ ]:
from dataclasses import dataclass, field
from typing import Tuple
import os

@dataclass
class Cfg:
    # READ data from the original run; WRITE checkpoints somewhere separate so the
    # OPSD notebook's files are never touched.
    data_root: str  = "/content/drive/MyDrive/sdar_posttrain"
    drive_root: str = "/content/drive/MyDrive/sdar_superviseall"

    model_id: str = "JetLM/SDAR-4B-Chat-b32"
    block_size: int = 32

    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    lora_targets: Tuple[str, ...] = ("q_proj","k_proj","v_proj","o_proj",
                                     "gate_proj","up_proj","down_proj")

    # ---- the curriculum: same objective, three schedules ----
    schedules: Tuple[int, ...] = (4, 2, 1)     # tokens per step, coarse -> fine
    epochs: int = 2
    supervise_all: bool = True                 # ALL block positions, every sub-step

    max_target_tokens: int = 384   # BIGGEST time lever — see the projection below
    gen_budget: int = 1536         # for benchmarking only
    lr: float = 1e-5
    grad_accum: int = 4
    warmup_steps: int = 10
    ckpt_every_samples: int = 20
    repeat4_max: float = 0.90

    bench_schedules: Tuple[int, ...] = (2, 1)  # evaluate at both, matching earlier runs

cfg = Cfg()
for sub in ("ckpt","results"):
    os.makedirs(os.path.join(cfg.drive_root, sub), exist_ok=True)
STATE_PATH = os.path.join(cfg.drive_root, "curriculum_state.json")
def ck(n): return os.path.join(cfg.drive_root, "ckpt", n)
def rs(n): return os.path.join(cfg.drive_root, "results", n)

N_TRAIN = 500
print(f"data from : {cfg.data_root}")
print(f"writing to: {cfg.drive_root}   (separate — OPSD notebook unaffected)\n")
tot = 0
print(f"{'stage':<20}{'tok/step':>9}{'forwards':>12}{'hours':>8}")
for i, per in enumerate(cfg.schedules):
    f = N_TRAIN*cfg.epochs*(cfg.max_target_tokens//per); tot += f
    print(f"{'S'+str(i+1):<20}{per:>9}{f:>12,}{f*0.055/3600:>8.1f}")
print(f"{'TOTAL':<20}{'':>9}{tot:>12,}{tot*0.055/3600:>8.1f}")
print(f"\nmax_target_tokens={cfg.max_target_tokens}: halving it ~halves every stage.")


## 4 · `flash_attn` → pure-PyTorch replacements

In [ ]:
import os, sys, types, importlib, importlib.abc, importlib.machinery, importlib.util
import torch, torch.nn.functional as F

USE_FLASH_ATTN = False
for _n in [n for n in list(sys.modules) if n=="flash_attn" or n.startswith("flash_attn.")]:
    del sys.modules[_n]
for _n in [n for n in list(sys.modules) if "modeling_sdar" in n]:
    del sys.modules[_n]

import transformers.dynamic_module_utils as _dmu
_real = getattr(_dmu, "_orig_get_imports", _dmu.get_imports)
_dmu._orig_get_imports = _real
def _pgi(fn):
    imp = list(_real(fn))
    if "flash_attn" in imp and os.path.basename(str(fn)).startswith("modeling_"):
        imp = [i for i in imp if i != "flash_attn"]
    return imp
_dmu.get_imports = _pgi

import transformers.utils as _tu
for _old, _c in {"LossKwargs": ["TransformersKwargs"]}.items():
    if not hasattr(_tu, _old):
        v = None
        for cand in _c:
            for mp in ("transformers.utils","transformers.processing_utils",
                       "transformers.modeling_utils","transformers"):
                try:
                    m = importlib.import_module(mp)
                    if hasattr(m, cand): v = getattr(m, cand); break
                except Exception: pass
            if v is not None: break
        setattr(_tu, _old, v if v is not None else type(_old, (dict,), {}))

def _rms_norm_fn(x, weight, bias=None, residual=None, x1=None, weight1=None, bias1=None,
                 eps=1e-6, dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                 zero_centered_weight=False, return_dropout_mask=False, out_dtype=None,
                 out=None, residual_out=None):
    xdt = x.dtype
    if x1 is not None: x = x + x1
    base = ((x.float()+residual.float()) if residual_in_fp32 else (x+residual)) if residual is not None \
           else (x.float() if residual_in_fp32 else x)
    nr = base; xf = base.float()
    y = (xf*torch.rsqrt(xf.pow(2).mean(-1,keepdim=True)+eps)).to(xdt) * \
        ((1.0+weight) if zero_centered_weight else weight)
    if bias is not None: y = y + bias
    return (y, (nr if residual_in_fp32 else nr.to(xdt))) if prenorm else y

def _layer_norm_fn(x, weight, bias=None, residual=None, x1=None, weight1=None, bias1=None,
                   eps=1e-6, dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                   zero_centered_weight=False, is_rms_norm=False, return_dropout_mask=False,
                   out_dtype=None, out=None, residual_out=None):
    if is_rms_norm:
        return _rms_norm_fn(x, weight, bias, residual, x1, weight1, bias1, eps, dropout_p,
                            rowscale, prenorm, residual_in_fp32, zero_centered_weight,
                            return_dropout_mask, out_dtype, out, residual_out)
    xdt = x.dtype
    if x1 is not None: x = x + x1
    base = ((x.float()+residual.float()) if residual_in_fp32 else (x+residual)) if residual is not None \
           else (x.float() if residual_in_fp32 else x)
    nr = base; xf = base.float(); mu = xf.mean(-1,keepdim=True)
    y = ((xf-mu)*torch.rsqrt((xf-mu).pow(2).mean(-1,keepdim=True)+eps)).to(xdt) * \
        ((1.0+weight) if zero_centered_weight else weight)
    if bias is not None: y = y + bias
    return (y, (nr if residual_in_fp32 else nr.to(xdt))) if prenorm else y

def _expand_kv(k,v,nq):
    nk = k.shape[-2]
    if nk != nq:
        r = nq//nk; k = k.repeat_interleave(r,dim=-2); v = v.repeat_interleave(r,dim=-2)
    return k,v
def _flash_attn_func(q,k,v,dropout_p=0.0,softmax_scale=None,causal=False,window_size=(-1,-1),
                     softcap=0.0,alibi_slopes=None,deterministic=False,return_attn_probs=False,**kw):
    k,v=_expand_kv(k,v,q.shape[-2])
    o=F.scaled_dot_product_attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),
                                     is_causal=causal,scale=softmax_scale,dropout_p=0.0)
    return o.transpose(1,2)
def _flash_attn_qkvpacked_func(qkv,**kw):
    q,k,v=qkv.unbind(dim=2); return _flash_attn_func(q,k,v,**kw)
def _flash_attn_varlen_func(q,k,v,cu_seqlens_q,cu_seqlens_k,max_seqlen_q=None,max_seqlen_k=None,
                            dropout_p=0.0,softmax_scale=None,causal=False,**kw):
    cq,ck_=cu_seqlens_q.tolist(),cu_seqlens_k.tolist(); outs=[]
    for i in range(len(cq)-1):
        qi,ki,vi=q[cq[i]:cq[i+1]],k[ck_[i]:ck_[i+1]],v[ck_[i]:ck_[i+1]]
        ki,vi=_expand_kv(ki,vi,qi.shape[-2])
        oi=F.scaled_dot_product_attention(qi.transpose(0,1).unsqueeze(0),ki.transpose(0,1).unsqueeze(0),
                                          vi.transpose(0,1).unsqueeze(0),is_causal=causal,
                                          scale=softmax_scale,dropout_p=0.0)
        outs.append(oi.squeeze(0).transpose(0,1))
    return torch.cat(outs,0)
def _pad_input(hs,idx,b,s):
    out=hs.new_zeros(b*s,hs.shape[-1]); out[idx]=hs; return out.view(b,s,-1)
def _unpad_input(hs,am,*a,**k):
    sl=am.sum(-1).to(torch.int32); idx=torch.nonzero(am.flatten(),as_tuple=False).flatten()
    h=hs.reshape(-1,hs.shape[-1])[idx]; cu=torch.zeros(sl.numel()+1,dtype=torch.int32,device=hs.device)
    cu[1:]=torch.cumsum(sl,0); return h,idx,cu,int(sl.max().item())
def _index_first_axis(x,idx): return x.reshape(-1,*x.shape[1:])[idx]

class _RMSNormModule(torch.nn.Module):
    def __init__(self,hidden_size,eps=1e-6,**kw):
        super().__init__(); self.weight=torch.nn.Parameter(torch.ones(hidden_size)); self.eps=eps
    def forward(self,x,residual=None,prenorm=False,**kw):
        return _rms_norm_fn(x,self.weight,None,residual=residual,eps=self.eps,prenorm=prenorm)

_REG={"rms_norm_fn":_rms_norm_fn,"layer_norm_fn":_layer_norm_fn,"RMSNorm":_RMSNormModule,
      "LayerNorm":torch.nn.LayerNorm,"flash_attn_func":_flash_attn_func,
      "flash_attn_qkvpacked_func":_flash_attn_qkvpacked_func,
      "flash_attn_varlen_func":_flash_attn_varlen_func,"pad_input":_pad_input,
      "unpad_input":_unpad_input,"index_first_axis":_index_first_axis}
def _uns(n):
    def f(*a,**k): raise RuntimeError(f"flash_attn.{n} has no shim but was CALLED — report it.")
    return f
class _FM(types.ModuleType):
    def __getattr__(self,n):
        if n in _REG: return _REG[n]
        if n.startswith("__"): raise AttributeError(n)
        return _uns(n)
class _FF(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self,fn,path=None,target=None):
        if fn=="flash_attn" or fn.startswith("flash_attn."):
            return importlib.machinery.ModuleSpec(fn,self,is_package=True)
    def create_module(self,spec):
        m=_FM(spec.name); m.__spec__=spec; m.__path__=[]; m.__version__="0.0-shim"; return m
    def exec_module(self,m): pass

try:
    import flash_attn; USE_FLASH_ATTN=True; print("Real flash_attn present.")
except ImportError:
    if not any(isinstance(f,_FF) for f in sys.meta_path): sys.meta_path.insert(0,_FF())
    import flash_attn; print("✅ pure-PyTorch flash_attn replacements active.")
_t=torch.randn(2,4,8); _w=torch.randn(8)
assert torch.allclose(_rms_norm_fn(_t,_w,eps=1e-6),
                      _t*torch.rsqrt(_t.pow(2).mean(-1,keepdim=True)+1e-6)*_w, atol=1e-5)
print("✅ RMSNorm replacement matches reference math.")


## 5 · Metrics

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

def repeat4(text):
    t = text.split()
    if len(t) < 4: return 0.0
    g = [tuple(t[i:i+4]) for i in range(len(t)-3)]
    return 1.0 - len(set(g))/len(g)

def extract_boxed(text):
    if not text: return None
    i = text.rfind("\\boxed")
    if i == -1: return None
    j = text.find("{", i)
    if j == -1: return None
    d = 0
    for k in range(j, len(text)):
        if text[k] == "{": d += 1
        elif text[k] == "}":
            d -= 1
            if d == 0: return text[j+1:k]
    return None

def has_complete_box(t): return extract_boxed(t) is not None

def _norm(s):
    if s is None: return None
    s = str(s).strip().replace(" ","")
    for a,b in (("\\left",""),("\\right",""),("\\dfrac","\\frac"),("\\tfrac","\\frac"),("$","")):
        s = s.replace(a,b)
    if s.startswith("\\text{") and s.endswith("}"): s = s[6:-1]
    return s

def answers_match(pred, gold):
    a,b = _norm(pred), _norm(gold)
    if a is None or b is None: return False
    if a == b: return True
    try: return abs(float(a)-float(b)) < 1e-6
    except Exception: return False

def structure_metrics(gen, ref, n_chunks=6):
    w = gen.split()
    if len(w) < n_chunks*4 or not ref.strip(): return float("nan"),float("nan"),float("nan")
    chunks = [" ".join(c) for c in np.array_split(np.array(w), n_chunks)]
    rw = ref.split(); tail = " ".join(rw[-max(20,len(rw)//4):])
    corpus = chunks + [tail, ref, gen]
    try: V = TfidfVectorizer(ngram_range=(1,2), min_df=1).fit_transform(corpus)
    except ValueError: return float("nan"),float("nan"),float("nan")
    C,tl,fr,fg = V[:n_chunks],V[n_chunks],V[n_chunks+1],V[n_chunks+2]
    sims = cosine_similarity(C,tl).ravel()
    prog = spearmanr(np.arange(n_chunks),sims).correlation if float(np.std(sims))>1e-9 else 0.0
    if prog is None or np.isnan(prog): prog = 0.0
    P = cosine_similarity(C); iu = np.triu_indices(n_chunks,k=1)
    return float(prog), float(P[iu].mean()), float(cosine_similarity(fg,fr)[0,0])

assert extract_boxed(r"a \boxed{1}, b \boxed{\frac{1}{2}}") == r"\frac{1}{2}"
assert answers_match(r"\dfrac{1}{2}", r"\frac{1}{2}") and not answers_match("3","4")
assert repeat4("a b c d "*6) > 0.5
print("✅ metric self-tests pass")


## 6 · Data — 500 problems = `stage_a + stage_b + stage_c` from the original partitions
The OPSD pool and the 40-problem benchmark are deliberately **not** used for training here, so
this experiment stays comparable to earlier results and can run alongside the OPSD notebook.

In [ ]:
import json, os
from collections import Counter

DATA_PATH = os.path.join(cfg.data_root, "data_partitions.json")
assert os.path.isfile(DATA_PATH), f"{DATA_PATH} not found — check cfg.data_root."
D = json.load(open(DATA_PATH))

train_pool = D["stage_a"] + D["stage_b"] + D["stage_c"]
bench = D["bench"]
print(f"train pool: {len(train_pool)} problems  {dict(Counter(p['tier'] for p in train_pool))}")
print(f"benchmark : {len(bench)} problems  {dict(Counter(p['tier'] for p in bench))}")

bq = {p["question"] for p in bench}
ov = sum(1 for p in train_pool if p["question"] in bq)
assert ov == 0, "benchmark leaked into training data!"
print(f"train/benchmark overlap: {ov} ✓")
if len(train_pool) < N_TRAIN:
    print(f"⚠️ only {len(train_pool)} available (wanted {N_TRAIN}) — proceeding with what's there.")
train_pool = train_pool[:N_TRAIN]


## 7 · Model, LoRA, mask
`modeling_sdar.py`'s outer `forward()` has `_update_causal_mask` commented out, so the block-causal
mask is entirely our responsibility. `fuse_cross_entropy` is forced off — it returns `logits=None`
whenever `self.training` is True.

In [ ]:
import torch, contextlib, gc, math, time, transformers
from packaging import version as _v
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel

_dt_kw = "dtype" if _v.parse(transformers.__version__.split("+")[0]) >= _v.parse("4.56.0") \
         else "torch_dtype"

def build_block_causal_mask(seq_len, prompt_len, block_size, device):
    idx = torch.arange(seq_len, device=device)
    is_resp = idx >= prompt_len
    blk = torch.where(is_resp, (idx-prompt_len)//block_size, idx)
    qr,kr = is_resp.unsqueeze(1), is_resp.unsqueeze(0)
    qi,ki = idx.unsqueeze(1), idx.unsqueeze(0)
    qb,kb = blk.unsqueeze(1), blk.unsqueeze(0)
    return ((~qr)&(ki<=qi)) | (qr&(~kr)) | (qr&kr&(kb<=qb))

def load_base():
    tok = AutoTokenizer.from_pretrained(cfg.model_id, trust_remote_code=True)
    m = AutoModelForCausalLM.from_pretrained(
        cfg.model_id, trust_remote_code=True, device_map="cuda",
        attn_implementation="flash_attention_2" if USE_FLASH_ATTN else "sdpa",
        **{_dt_kw: torch.bfloat16})
    if hasattr(m.config,"fuse_cross_entropy"): m.config.fuse_cross_entropy = False
    m.gradient_checkpointing_disable()
    return m, tok

def fresh_lora(m):
    return get_peft_model(m, LoraConfig(
        r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.lora_targets), bias="none", task_type="CAUSAL_LM"))

def build_invalid(m, tok):
    vs, rv = m.config.vocab_size, len(tok)
    inv = torch.zeros(vs, dtype=torch.bool)
    if rv < vs: inv[rv:] = True
    mid = getattr(tok,"mask_token_id",None) or 151669
    inv[mid] = True
    return inv.to(m.device), mid

def student_prompt(tok, q):
    return tok.apply_chat_template(
        [{"role":"user","content": f"{q}\nPlease reason step by step, and put your final answer "
                                   f"within \\boxed{{}}."}], tokenize=False, add_generation_prompt=True)

def teacher_prompt(tok, q, sol):
    c = (f"{q}\n\nHere is a reference solution:\n{sol}\n\nAfter understanding the reference "
         f"solution, solve the problem yourself.\nPlease reason step by step, and put your final "
         f"answer within \\boxed{{}}.")
    return tok.apply_chat_template([{"role":"user","content":c}], tokenize=False,
                                   add_generation_prompt=True)

def logits_of(model, ids, prompt_len, teacher=False):
    mask = build_block_causal_mask(ids.shape[1], prompt_len, cfg.block_size, ids.device)[None,None]
    ctx = model.disable_adapter() if teacher else contextlib.nullcontext()
    with ctx:
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            return model(input_ids=ids, attention_mask=mask).logits

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()
    print(f"    VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

def save_adapters(model, name):
    d = ck(name); os.makedirs(d, exist_ok=True); model.save_pretrained(d)

def adapters_exist(name):
    return os.path.isfile(os.path.join(ck(name),"adapter_model.safetensors")) or \
           os.path.isfile(os.path.join(ck(name),"adapter_model.bin"))

DEFAULT_STATE = {s:{"done":False,"epoch":0,"sample":0,"opt_step":0}
                 for s in ("stage_a","stage_b","stage_c","opsd1","opsd2","bench")}
def load_state():
    if os.path.isfile(STATE_PATH):
        st = json.load(open(STATE_PATH))
        for k,v in DEFAULT_STATE.items(): st.setdefault(k, dict(v))
        return st
    return {k:dict(v) for k,v in DEFAULT_STATE.items()}
def save_state(st): json.dump(st, open(STATE_PATH,"w"), indent=1)

STATE = load_state()
print("model helpers ready.")


## 8 · State

In [ ]:
import json, os, torch

STAGE_NAMES = [f"s{i+1}_t{per}" for i, per in enumerate(cfg.schedules)]
DEFAULT_STATE = {s: {"done": False, "epoch": 0, "sample": 0, "opt_step": 0} for s in STAGE_NAMES}

def load_state():
    if os.path.isfile(STATE_PATH):
        st = json.load(open(STATE_PATH))
        for k, v in DEFAULT_STATE.items(): st.setdefault(k, dict(v))
        return st
    return {k: dict(v) for k, v in DEFAULT_STATE.items()}
def save_state(st): json.dump(st, open(STATE_PATH, "w"), indent=1)
def save_adapters(model, name):
    d = ck(name); os.makedirs(d, exist_ok=True); model.save_pretrained(d)
def adapters_exist(name):
    return os.path.isfile(os.path.join(ck(name), "adapter_model.safetensors")) or \
           os.path.isfile(os.path.join(ck(name), "adapter_model.bin"))

STATE = load_state()
print("STAGES:", STAGE_NAMES)
for k, v in STATE.items():
    print(f"  {k:<10} done={str(v['done']):<5} epoch={v['epoch']} sample={v['sample']}")
print("\ncheckpoints:", sorted(os.listdir(ck(""))) if os.path.isdir(ck("")) else [])


## 9 · SFT engine — `supervise_all`, backward **per sub-step**

Backward runs once per sub-step rather than once per block. That is mathematically identical
(gradients accumulate across `backward()` calls) but caps peak memory at one forward graph instead
of `32/tokens_per_step` of them — the 1-token/step stage would hold 32 simultaneously otherwise,
which is what caused the earlier OOM.

Each sub-step's loss is divided by the block's total supervised position-count so the result matches
a single per-block backward exactly. With `supervise_all=True` that count is simply `bs × n_sub`,
since every position is supervised at every sub-step.

In [ ]:
import torch, torch.nn.functional as F, math, time, json
from transformers import get_cosine_schedule_with_warmup

def sft_sequence_loss(model, tok, invalid, mask_id, question, solution,
                      tokens_per_step, loss_scale, supervise_all):
    dev = model.device
    p_ids = tok(student_prompt(tok, question), return_tensors="pt").input_ids.to(dev)
    s_ids = tok(solution, return_tensors="pt", add_special_tokens=False).input_ids.to(dev)
    if tok.eos_token_id is not None:
        s_ids = torch.cat([s_ids, torch.tensor([[tok.eos_token_id]], device=dev,
                                               dtype=s_ids.dtype)], dim=1)
    s_ids = s_ids[:, :cfg.max_target_tokens]
    B, per = cfg.block_size, tokens_per_step
    prompt_len = p_ids.shape[1]
    ctx = p_ids
    running, npos, nblk = 0.0, 0, 0

    for b0 in range(0, s_ids.shape[1], B):
        gold = s_ids[:, b0:b0+B]; bs = gold.shape[1]
        if bs == 0: break
        n_sub = math.ceil(bs/per)
        total_bpos = bs*n_sub if supervise_all else (n_sub*bs - per*n_sub*(n_sub-1)//2)
        revealed = torch.full((1,bs), mask_id, device=dev, dtype=ctx.dtype)
        still = torch.ones(bs, dtype=torch.bool, device=dev)
        nblk += 1
        while bool(still.any()):
            work = torch.cat([ctx, revealed], dim=1)
            pos = torch.arange(ctx.shape[1], ctx.shape[1]+bs, device=dev)
            lg = logits_of(model, work, prompt_len)[0, pos, :].float().masked_fill(invalid, float("-inf"))
            sup = torch.arange(bs, device=dev) if supervise_all else still.nonzero(as_tuple=True)[0]
            step_loss = F.cross_entropy(lg[sup], gold[0, sup], reduction="sum") / total_bpos
            (step_loss * loss_scale).backward()
            running += step_loss.item(); npos += int(sup.numel())
            conf = torch.log_softmax(lg,-1).max(-1).values.masked_fill(~still, float("-inf"))
            k = min(per, int(still.sum().item()))
            sel = conf.topk(k).indices
            revealed[0, sel] = gold[0, sel]      # teacher-force GOLD
            still[sel] = False
        ctx = torch.cat([ctx, gold.detach()], dim=1)
    return running/max(1,nblk), npos, nblk

def run_stage(stage, source, tokens_per_step):
    """Independent per-stage runner: own model load, own optimizer, own cleanup."""
    st = STATE[stage]
    if st["done"]:
        print(f"{stage}: already done — skipping."); return
    resume = f"{stage}_inprogress" if adapters_exist(f"{stage}_inprogress") else source
    base, tok = load_base()
    if resume is None:
        print(f"{stage}: fresh LoRA"); model = fresh_lora(base)
    else:
        assert adapters_exist(resume), f"checkpoint '{resume}' missing"
        print(f"{stage}: loading adapters from {resume}")
        model = PeftModel.from_pretrained(base, ck(resume), is_trainable=True)
    model.train(); model.print_trainable_parameters()
    invalid, mask_id = build_invalid(model, tok)
    trainable = [q for q in model.parameters() if q.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=cfg.lr)
    sched = get_cosine_schedule_with_warmup(
        opt, cfg.warmup_steps, max(1, math.ceil(len(train_pool)*cfg.epochs/cfg.grad_accum)))
    for _ in range(st["opt_step"]): sched.step()
    opt.zero_grad(set_to_none=True)

    print(f"\n=== {stage} | {len(train_pool)} problems x {cfg.epochs} epochs | "
          f"{tokens_per_step} tok/step | supervise_all={cfg.supervise_all} ===")
    if st["epoch"] or st["sample"]:
        print(f"  resuming at epoch {st['epoch']+1}, sample {st['sample']}")
    acc, t0 = 0, time.time()
    for ep in range(st["epoch"], cfg.epochs):
        start = st["sample"] if ep == st["epoch"] else 0
        for i in range(start, len(train_pool)):
            p = train_pool[i]
            loss, npos, nblk = sft_sequence_loss(model, tok, invalid, mask_id,
                                                 p["question"], p["solution"], tokens_per_step,
                                                 1.0/cfg.grad_accum, cfg.supervise_all)
            acc += 1
            if acc == cfg.grad_accum:
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
                acc = 0; st["opt_step"] += 1
            st["epoch"], st["sample"] = ep, i+1
            save_state(STATE)
            with open(rs(f"{stage}_log.jsonl"), "a") as f:
                f.write(json.dumps({"ep":ep,"i":i,"loss":loss,"npos":npos,"nblk":nblk})+"\n")
            if (i+1) % 20 == 0:
                el = (time.time()-t0)/60
                print(f"  e{ep+1} [{i+1:>4}/{len(train_pool)}] loss {loss:.4f} | {nblk:>2} blk "
                      f"{npos:>5} sup | {el:.1f} min | "
                      f"{torch.cuda.memory_allocated()/1e9:.1f} GB")
            if (i+1) % cfg.ckpt_every_samples == 0:
                save_adapters(model, f"{stage}_inprogress")
        st["sample"] = 0; st["epoch"] = ep+1
        save_state(STATE); save_adapters(model, f"{stage}_inprogress")
        print(f"  -- epoch {ep+1}/{cfg.epochs} done --")
    if acc > 0:
        torch.nn.utils.clip_grad_norm_(trainable, 1.0); opt.step(); sched.step()
    st["done"] = True; save_state(STATE); save_adapters(model, stage)
    print(f"✅ {stage} complete -> {ck(stage)}")
    del model, base, tok, invalid, opt, sched, trainable
    import gc; gc.collect(); torch.cuda.empty_cache()
    print(f"    VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print("SFT engine ready (supervise_all, per-sub-step backward).")


## 10 · Stage S1 — 4 tokens/step
Starts from a fresh LoRA on the raw base. Independent cell: own model load, own optimizer, own cleanup.

In [ ]:
run_stage("s1_t4", None, 4)


## 11 · Stage S2 — 2 tokens/step
Resumes from S1. Independent cell: own model load, own optimizer, own cleanup.

In [ ]:
run_stage("s2_t2", "s1_t4", 2)


## 12 · Stage S3 — 1 token/step
Resumes from S2. Independent cell: own model load, own optimizer, own cleanup.

In [ ]:
run_stage("s3_t1", "s2_t2", 1)


## 13 · Benchmark — raw vs S1 vs S2 vs S3
Evaluated at **2 and 1 tokens/step** on the held-out 40, with the corrected scorer (case-insensitive,
plus a conclusion-phrase fallback when `\boxed{}` is absent). Reports per-difficulty-tier breakdowns,
because the earlier finding was that SFT helped on easy problems and hurt on medium/hard — the
overall number alone hides that.

In [ ]:
import torch, time, json, re

def score_generation(text, gold):
    def norm_ci(s):
        if s is None: return None
        s = str(s).strip().lower().replace(" ", "")
        for a,b in (("\\left",""),("\\right",""),("\\dfrac","\\frac"),("\\tfrac","\\frac"),("$","")):
            s = s.replace(a,b)
        if s.startswith("\\text{") and s.endswith("}"): s = s[6:-1]
        return s
    def match(a,b):
        a,b = norm_ci(a), norm_ci(b)
        if a is None or b is None: return False
        if a == b: return True
        try: return abs(float(a)-float(b)) < 1e-6
        except Exception: return False
    pred = extract_boxed(text)
    strict_ok = match(pred, gold) if pred else False
    if pred is None:
        tail = text[-300:]
        for pat in (r"(?:the answer is|answer:)\s*\$?([^\.\n$]{1,40})",
                    r"(?:the limit is|equals?)\s*\$?([^\.\n$]{1,40})",
                    r"=\s*\$?([^\.\n$=]{1,25})\s*\$?\.?\s*$"):
            m = re.findall(pat, tail, re.IGNORECASE)
            if m: pred = m[-1].strip(); break
    return {"pred": pred, "correct": match(pred, gold) if pred else False,
            "correct_strict": strict_ok}

@torch.no_grad()
def bench_rollout(model, tok, invalid, mask_id, question, tps):
    dev = model.device
    ids = tok(student_prompt(tok, question), return_tensors="pt").input_ids.to(dev)
    plen = ids.shape[1]; gen, txt, why = 0, "", "budget"
    while gen < cfg.gen_budget:
        cur = ids.shape[1]
        work = torch.cat([ids, torch.full((1,cfg.block_size), mask_id, dtype=ids.dtype, device=dev)], 1)
        pos = torch.arange(cur, cur+cfg.block_size, device=dev)
        still = torch.ones(cfg.block_size, dtype=torch.bool, device=dev)
        while bool(still.any()):
            lg = logits_of(model, work, plen)[0,pos,:].float().masked_fill(invalid, float("-inf"))
            conf = torch.log_softmax(lg,-1).max(-1).values.masked_fill(~still, float("-inf"))
            k = min(tps, int(still.sum().item()))
            sel = conf.topk(k).indices
            work[0,pos[sel]] = lg[sel].argmax(-1); still[sel] = False
        new = work[:,cur:cur+cfg.block_size]
        ids = torch.cat([ids,new],1); gen += cfg.block_size
        g = [t for t in ids[0,plen:].tolist() if t < len(tok)]
        txt = tok.decode(g, skip_special_tokens=True)
        if has_complete_box(txt): why="box"; break
        if tok.eos_token_id is not None and bool((new==tok.eos_token_id).any()): why="eos"; break
        if gen >= 128 and repeat4(txt) > cfg.repeat4_max: why="collapse"; break
    return txt, gen, why

def eval_model(label, model, tok):
    invalid, mask_id = build_invalid(model, tok); model.eval()
    out = {}
    for tps in cfg.bench_schedules:
        rows, t0 = [], time.time()
        for i, p in enumerate(bench):
            txt, n, why = bench_rollout(model, tok, invalid, mask_id, p["question"], tps)
            sc = score_generation(txt, p["gold"])
            rows.append({**sc, "tier": p["tier"], "repeat4": repeat4(txt), "stop": why})
            if (i+1) % 10 == 0:
                nc = sum(r["correct"] for r in rows)
                print(f"    {label} t{tps}: {i+1}/{len(bench)} | correct {nc}/{len(rows)} "
                      f"| {(time.time()-t0)/60:.1f} min")
        d = {"overall": sum(r["correct"] for r in rows)/len(rows),
             "strict": sum(r["correct_strict"] for r in rows)/len(rows),
             "rep4": sum(r["repeat4"] for r in rows)/len(rows)}
        for tier in ("easy","medium","hard"):
            tr = [r for r in rows if r["tier"]==tier]
            d[tier] = sum(r["correct"] for r in tr)/len(tr) if tr else float("nan")
        out[tps] = d
        print(f"  {label} t{tps}: overall {d['overall']:.0%} (strict {d['strict']:.0%}) | "
              f"easy {d['easy']:.0%} med {d['medium']:.0%} hard {d['hard']:.0%} | rep4 {d['rep4']:.3f}")
    return out

results = {}
print("=== RAW ===")
m, tok = load_base(); results["raw"] = eval_model("raw", m, tok)
del m, tok; import gc; gc.collect(); torch.cuda.empty_cache()

for stage in STAGE_NAMES:
    if not adapters_exist(stage):
        print(f"\n{stage}: no adapters — skipping"); continue
    print(f"\n=== {stage} ===")
    base, tok = load_base()
    peft = PeftModel.from_pretrained(base, ck(stage), is_trainable=False)
    m = peft.merge_and_unload(); m.gradient_checkpointing_disable()
    if hasattr(m.config,"fuse_cross_entropy"): m.config.fuse_cross_entropy = False
    results[stage] = eval_model(stage, m, tok)
    del m, peft, base, tok; gc.collect(); torch.cuda.empty_cache()

json.dump(results, open(rs("curriculum_benchmark.json"),"w"), indent=1)
print(f"\n{'='*76}\nSUMMARY")
for tps in cfg.bench_schedules:
    print(f"\n--- tokens_per_step = {tps} ---")
    print(f"{'model':<12}{'overall':>9}{'strict':>9}{'easy':>8}{'medium':>8}{'hard':>8}{'rep4':>8}")
    for name, r in results.items():
        if tps not in r: continue
        d = r[tps]
        print(f"{name:<12}{d['overall']:>9.0%}{d['strict']:>9.0%}{d['easy']:>8.0%}"
              f"{d['medium']:>8.0%}{d['hard']:>8.0%}{d['rep4']:>8.3f}")
print("\nwrote", rs("curriculum_benchmark.json"))


## 14 · Notes

**Run order.** §1–§9 each session, then §10/§11/§12 (S1→S2→S3), then §13. Each stage skips itself
if `STATE` says done and resumes mid-stage otherwise, so re-running the notebook is always safe.

**Judge by §13, not by training loss.** `supervise_all=True` dilutes the loss with near-zero
copy-through terms from already-revealed positions, so its value is not comparable to a masked-only
stage — and comparing S1/S2/S3 losses to each other is also unsafe, since the number of supervised
positions per block differs by schedule.

**The comparison that matters** is the per-tier breakdown against `raw`. The earlier pipeline helped
on easy problems and hurt on medium/hard, with repeat-4 rising at every stage. If that pattern
repeats here, the problem is the `supervise_all` objective itself rather than the specific schedule
mix. If S1 helps and only S2/S3 hurt, the schedule (or cumulative drift) is the culprit.

**Watch repeat-4 across S1→S2→S3.** In the original run it rose monotonically (0.222 → 0.321 →
0.419 at 2 tok/step), tracking LoRA weight-norm growth. Whether that repeats here is the single most
diagnostic number in the table.